# Phase 5 — Extraction d'entités et exploitation du layout

**Objectif** : passer de la reconnaissance de texte brute à une extraction d'information
structurée (paires clé-valeur), en comparant deux approches :

1. **Règles + OCR** : détection de labels connus par mots-clés/regex directement dans le
   texte OCR (sans information spatiale).
2. **OCR + layout** : association question → réponse par **voisinage spatial**, en
   utilisant les bounding boxes des mots retournées par Tesseract.

**Champs ciblés** : d'après l'exploration de la Phase 1, FUNSD est composé de memos
d'entreprise américains — les libellés natifs les plus fréquents sont `DATE`, `TO`,
`FROM`, `CC`, `SUBJECT`, `APPROVED BY`, `DIVISION NAME`. On extrait ces champs natifs
plutôt que les champs administratifs marocains initialement envisagés (cf. note de
synthèse de la Phase 1).

> Ce notebook est **autonome** : toutes les fonctions (OCR, extraction, évaluation)
> sont définies directement ici. Il détecte automatiquement l'emplacement du dataset,
> comme les notebooks 03 et 04.


In [ ]:
import json
import re
import platform
import os
from pathlib import Path
from collections import Counter

import cv2
import numpy as np
import pandas as pd
import pytesseract

if platform.system() == "Windows":
    for _path in [r"C:\Program Files\Tesseract-OCR\tesseract.exe",
                  r"C:\Program Files (x86)\Tesseract-OCR\tesseract.exe"]:
        if os.path.exists(_path):
            pytesseract.pytesseract.tesseract_cmd = _path
            break

HERE = Path.cwd()
print("Dossier courant (cwd) :", HERE.resolve())


## 0. Détection automatique du dataset

In [ ]:
def find_split_dir(candidates_roots, split_names):
    for root in candidates_roots:
        for name in split_names:
            candidate = root / name
            if (candidate / "images").is_dir() and (candidate / "annotations").is_dir():
                return candidate
    return None


def find_project_root(start, markers=("src", "dataset", "data"), max_levels=8):
    current = start
    for _ in range(max_levels):
        if any((current / m).is_dir() for m in markers):
            return current
        if current.parent == current:
            break
        current = current.parent
    return None


PROJECT_ROOT = find_project_root(HERE)
if PROJECT_ROOT is None:
    raise FileNotFoundError(f"Impossible de trouver la racine du projet depuis {HERE}.")
print("Racine du projet détectée :", PROJECT_ROOT.resolve())

POSSIBLE_ROOTS = [PROJECT_ROOT, PROJECT_ROOT / "dataset", PROJECT_ROOT / "data", HERE, HERE.parent]
POSSIBLE_ROOTS = [r for r in POSSIBLE_ROOTS if r.exists()]

RAW_DIR = find_split_dir(POSSIBLE_ROOTS, ["training_data", "raw"])
if RAW_DIR is None:
    raise FileNotFoundError("Dossier d'entraînement (training_data/raw) introuvable.")
print("Dossier train détecté :", RAW_DIR.resolve())


def build_manifest(split_dir):
    images = sorted((split_dir / "images").glob("*.png"))
    manifest = []
    for img in images:
        ann = split_dir / "annotations" / f"{img.stem}.json"
        if ann.exists():
            manifest.append({"image": img.name, "annotation": ann.name})
    return manifest


manifest = build_manifest(RAW_DIR)
print(f"{len(manifest)} documents FUNSD (train set complet)")


## 1. Fonction OCR avec bounding boxes (autonome)

In [ ]:
def run_ocr_with_boxes(img, lang="eng"):
    """OCR avec positions des mots (nécessaire pour l'approche layout)."""
    rgb = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB) if len(img.shape) == 2 else cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    text = pytesseract.image_to_string(rgb, lang=lang).strip()
    data = pytesseract.image_to_data(rgb, lang=lang, output_type=pytesseract.Output.DICT)

    words = []
    for i in range(len(data["text"])):
        w = data["text"][i].strip()
        if not w:
            continue
        conf = float(data["conf"][i]) if str(data["conf"][i]) not in ("-1",) else -1.0
        x, y, bw, bh = data["left"][i], data["top"][i], data["width"][i], data["height"][i]
        words.append({"text": w, "box": [x, y, x + bw, y + bh], "conf": conf})

    return {"text": text, "words": words}


## 2. Champs cibles et vérité terrain

On construit, pour chaque document, les paires question → réponse **de référence**
(ground truth), à partir du champ `linking` de l'annotation FUNSD — c'est ce à quoi
on comparera les extractions automatiques.

In [ ]:
TARGET_FIELDS = {
    "DATE": re.compile(r"^date\b", re.IGNORECASE),
    "TO": re.compile(r"^to\s*:?$", re.IGNORECASE),
    "FROM": re.compile(r"^from\s*:?$", re.IGNORECASE),
    "CC": re.compile(r"^cc\s*:?$", re.IGNORECASE),
    "SUBJECT": re.compile(r"^subject\b", re.IGNORECASE),
    "APPROVED_BY": re.compile(r"^approved\s*by\b", re.IGNORECASE),
    "DIVISION_NAME": re.compile(r"^division\s*name\b", re.IGNORECASE),
}


def ground_truth_pairs(annotation):
    """Reconstruit les paires {champ: valeur} de référence via le linking FUNSD."""
    by_id = {item["id"]: item for item in annotation["form"]}
    pairs = {}
    for item in annotation["form"]:
        if item["label"] != "question":
            continue
        field_name = None
        for name, pattern in TARGET_FIELDS.items():
            if pattern.search(item["text"].strip()):
                field_name = name
                break
        if field_name is None:
            continue
        for (src, dst) in item.get("linking", []):
            target = by_id.get(dst)
            if target and target["label"] == "answer" and target["text"].strip():
                pairs[field_name] = target["text"].strip()
    return pairs


# aperçu sur quelques documents
sample_gt = []
for entry in manifest[:5]:
    ann = json.loads((RAW_DIR / "annotations" / entry["annotation"]).read_text(encoding="utf-8"))
    gt = ground_truth_pairs(ann)
    sample_gt.append({"document": entry["image"], **gt})

pd.DataFrame(sample_gt)


## 3. Approche 1 — Règles + OCR (texte brut, sans layout)

On cherche, dans le texte OCR complet (une seule chaîne), les motifs `LABEL: VALEUR`
via une expression régulière — sans utiliser aucune position spatiale.

In [ ]:
RULE_PATTERNS = {
    "DATE": re.compile(r"date\s*:?\s*([0-9/\-\. ]{6,12})", re.IGNORECASE),
    "TO": re.compile(r"\bto\s*:\s*([A-Za-z.,' \-]{3,40})", re.IGNORECASE),
    "FROM": re.compile(r"\bfrom\s*:\s*([A-Za-z.,' \-]{3,40})", re.IGNORECASE),
    "CC": re.compile(r"\bcc\s*:\s*([A-Za-z.,' \-]{3,40})", re.IGNORECASE),
    "SUBJECT": re.compile(r"subject\s*:?\s*([A-Za-z0-9.,' \-]{3,50})", re.IGNORECASE),
    "APPROVED_BY": re.compile(r"approved\s*by\s*:?\s*([A-Za-z.,' \-]{3,40})", re.IGNORECASE),
    "DIVISION_NAME": re.compile(r"division\s*name\s*:?\s*([A-Za-z.,' \-]{3,40})", re.IGNORECASE),
}


def extract_rule_based(ocr_text):
    """Approche 1 : cherche 'LABEL: valeur' directement dans le texte OCR concaténé."""
    results = {}
    for field, pattern in RULE_PATTERNS.items():
        m = pattern.search(ocr_text)
        if m:
            value = m.group(1).strip().rstrip(".,;")
            if value:
                results[field] = value
    return results


## 4. Approche 2 — OCR + layout (voisinage spatial)

On détecte le mot-label dans la sortie OCR (avec sa bounding box), puis on cherche le
ou les mots les plus proches **à droite, sur la même ligne** — sans utiliser de regex
sur la valeur, uniquement la position.

In [ ]:
LABEL_KEYWORDS = {
    "DATE": "date", "TO": "to", "FROM": "from", "CC": "cc",
    "SUBJECT": "subject", "APPROVED_BY": "approved", "DIVISION_NAME": "division",
}


def _distance(b1, b2):
    c1 = ((b1[0] + b1[2]) / 2, (b1[1] + b1[3]) / 2)
    c2 = ((b2[0] + b2[2]) / 2, (b2[1] + b2[3]) / 2)
    return ((c1[0] - c2[0]) ** 2 + (c1[1] - c2[1]) ** 2) ** 0.5


def extract_layout_based(ocr_words, max_value_words=4):
    """Approche 2 : associe chaque label détecté au(x) mot(s) le(s) plus proche(s)
    à droite, sur la même bande horizontale."""
    results = {}
    used = set()

    for i, w in enumerate(ocr_words):
        clean = w["text"].rstrip(":").strip().lower()
        matched_field = None
        for field, kw in LABEL_KEYWORDS.items():
            if clean == kw or clean.startswith(kw):
                matched_field = field
                break
        if matched_field is None or matched_field in results:
            continue

        qx0, qy0, qx1, qy1 = w["box"]
        candidates = []
        for j, w2 in enumerate(ocr_words):
            if j == i or j in used:
                continue
            vx0, vy0, vx1, vy1 = w2["box"]
            same_line = abs(((vy0 + vy1) / 2) - ((qy0 + qy1) / 2)) < (qy1 - qy0) * 1.6
            to_the_right = vx0 >= qx1 - 5
            if same_line and to_the_right:
                candidates.append((_distance(w["box"], w2["box"]), j, w2["text"]))

        candidates.sort(key=lambda t: t[0])
        chosen = candidates[:max_value_words]
        value_tokens = [c[2] for c in chosen]
        for c in chosen:
            used.add(c[1])

        if value_tokens:
            results[matched_field] = " ".join(value_tokens).strip().rstrip(".,;")

    return results


## 5. Exécution des deux approches et comparaison à la vérité terrain

> Sous-échantillon de 15 documents pour un temps d'exécution raisonnable
> (chaque document nécessite un appel OCR avec extraction des bounding boxes).

In [ ]:
N_SAMPLE = 15
subset = manifest[:N_SAMPLE]

all_results = []
extraction_rows = []

for entry in subset:
    img_path = RAW_DIR / "images" / entry["image"]
    ann = json.loads((RAW_DIR / "annotations" / entry["annotation"]).read_text(encoding="utf-8"))
    gt = ground_truth_pairs(ann)

    img = cv2.imread(str(img_path))
    ocr_result = run_ocr_with_boxes(img)

    pred_rules = extract_rule_based(ocr_result["text"])
    pred_layout = extract_layout_based(ocr_result["words"])

    all_fields = set(gt) | set(pred_rules) | set(pred_layout)
    for field in sorted(all_fields):
        extraction_rows.append({
            "document": entry["image"], "field": field,
            "ground_truth": gt.get(field), "pred_rules": pred_rules.get(field),
            "pred_layout": pred_layout.get(field),
        })

df_extraction = pd.DataFrame(extraction_rows)

results_dir = PROJECT_ROOT / "results" / "tables"
results_dir.mkdir(parents=True, exist_ok=True)
df_extraction.to_csv(results_dir / "phase5_extraction_results.csv", index=False)
df_extraction.head(15)


## 6. Évaluation : exact match, partial match, missing, incorrect

In [ ]:
def classify_match(pred, gt):
    gt_missing = gt is None or (isinstance(gt, float) and pd.isna(gt))
    pred_missing = pred is None or (isinstance(pred, float) and pd.isna(pred))
    if gt_missing:
        return "hors_perimetre" if not pred_missing else "n/a"
    if pred_missing:
        return "missing"
    p, g = str(pred).strip().lower(), str(gt).strip().lower()
    if p == g:
        return "exact_match"
    if p in g or g in p:
        return "partial_match"
    return "incorrect"


for method in ["pred_rules", "pred_layout"]:
    df_extraction[f"status_{method}"] = df_extraction.apply(
        lambda row: classify_match(row[method], row["ground_truth"]), axis=1
    )

summary_rows = []
for method in ["pred_rules", "pred_layout"]:
    status_counts = df_extraction[f"status_{method}"].value_counts().to_dict()
    tp = status_counts.get("exact_match", 0)
    partial = status_counts.get("partial_match", 0)
    fp = status_counts.get("incorrect", 0)
    fn = status_counts.get("missing", 0)

    predicted_total = tp + partial + fp
    gt_total = tp + partial + fn
    precision = (tp + 0.5 * partial) / predicted_total if predicted_total else 0.0
    recall = (tp + 0.5 * partial) / gt_total if gt_total else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0

    summary_rows.append({
        "methode": "Règles + OCR" if method == "pred_rules" else "OCR + layout",
        "exact_match": tp, "partial_match": partial, "incorrect": fp, "missing": fn,
        "precision": round(precision, 3), "recall": round(recall, 3), "f1": round(f1, 3),
    })

df_summary = pd.DataFrame(summary_rows)
df_summary.to_csv(results_dir / "phase5_evaluation_summary.csv", index=False)
df_summary


## 7. Sortie structurée (JSON) par document — livrable de la Phase 5

In [ ]:
structured_output = {}
for entry in subset:
    doc_rows = df_extraction[df_extraction["document"] == entry["image"]]
    structured_output[entry["image"]] = {
        row["field"]: {
            "ground_truth": row["ground_truth"],
            "predicted_rules": row["pred_rules"],
            "predicted_layout": row["pred_layout"],
        }
        for _, row in doc_rows.iterrows()
    }

out_path = results_dir / "phase5_structured_extraction.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(structured_output, f, ensure_ascii=False, indent=2)

print(f"Sortie structurée sauvegardée : {out_path}")
list(structured_output.items())[0]


## 8. Résultats attendus de la Phase 5 — récapitulatif

- **Module d'extraction** : deux approches implémentées et comparées — règles/regex sur
  texte brut, et voisinage spatial sur bounding boxes OCR.
- **Sortie structurée** : `results/tables/phase5_extraction_results.csv` (détail par
  champ/document) et `phase5_structured_extraction.json` (format JSON par document).
- **Évaluation comparative** : `results/tables/phase5_evaluation_summary.csv` —
  precision/recall/F1 pour chacune des deux approches.
- **Constat attendu** : l'approche layout devrait généralement mieux performer que les
  règles pures sur du texte scanné bruité, car elle ne dépend pas d'un pattern regex
  rigide qui peut être cassé par une erreur OCR isolée.
- **Limite à documenter** : les deux approches restent simples (pas de modèle
  pré-entraîné) — une extension possible en LayoutLMv3 est esquissée mais non
  implémentée (ressources GPU nécessaires, cf. cahier des charges).
- **Prochaine étape (Phase 6)** : valider sémantiquement les valeurs extraites et leur
  attribuer un score de confiance.
